# Projeto SQL — Análise de Serviço de Livros

Durante a pandemia, o hábito de leitura cresceu e novas startups surgiram para atender leitores. Recebemos o banco de dados de um serviço concorrente — livros, autores, editoras, classificações e avaliações — para extrair insights que apoiem a proposta de um novo produto.

Este notebook responde a cinco perguntas de negócio via SQL, usando pandas apenas para executar consultas e exibir resultados.

## Passo 1 — Objetivos do estudo

Entender o catálogo e o comportamento dos usuários para embasar decisões de produto:

1. Dimensionar o catálogo moderno (livros após 01/01/2000).
2. Medir engajamento por livro (nº de avaliações e classificação média).
3. Identificar a editora mais relevante em livros >50 páginas.
4. Encontrar autores mais bem avaliados (massa crítica de ≥50 classificações).
5. Medir o engajamento dos usuários mais ativos (>50 livros classificados).

In [1]:
# importa bibliotecas
import pandas as pd
from sqlalchemy import create_engine
db_config = {
 'user': 'practicum_student', # username
 'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7', # password
 'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
 'port': 5432, # connection port
 'db': 'data-analyst-final-project-db' # the name of the database
 }
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],

db_config['pwd'],
db_config['host'],
db_config['port'],
db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [2]:
from sqlalchemy import text

def run_query(query):
    '''Executa uma consulta SQL e retorna o resultado como DataFrame.'''
    with engine.connect() as conn:
        return pd.read_sql(text(query), con=conn)

## Passo 2 — Conexão e exploração das tabelas

Amostra das cinco tabelas para confirmar tipos, chaves e granularidade antes de consultar.

In [3]:
for table in ['books', 'authors', 'publishers', 'ratings', 'reviews']:
    display(run_query(f'SELECT * FROM {table} LIMIT 5'))

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


## Passo 3 — Livros lançados após 01/01/2000

**Tarefa:** contar quantos livros foram publicados depois de 01/01/2000.
**Abordagem:** `COUNT` sobre `books`, filtrando `publication_date` com `WHERE`.

In [4]:
query = '''
    SELECT COUNT(book_id) AS qtd_livros
    FROM books WHERE publication_date > '2000-01-01'
    '''
run_query(query)

,qtd_livros
0,819


**Resultado e conclusão:** 819 livros. O catálogo é fortemente concentrado em publicações recentes — a base atende bem o leitor de lançamentos, não o de clássicos. Interpretei "depois de" como `>` (exclusivo); ver Diário de Decisões.

## Passo 4 — Avaliações e classificação média por livro

**Tarefa:** para cada livro, o número de avaliações (reviews) e a classificação média (ratings).
**Abordagem:** `books` como base, `LEFT JOIN` com `reviews` e `ratings`. `COUNT(DISTINCT review_id)` evita a inflação do produto cartesiano; `AVG(rating)` não sofre com a duplicação uniforme.

In [5]:
query = '''
    SELECT
        b.book_id,
        b.title,
        COUNT(DISTINCT rev.review_id) AS qtd_avaliacoes,
        AVG(rat.rating) AS classificacao_media
    FROM books AS b
    LEFT JOIN reviews AS rev ON b.book_id = rev.book_id
    LEFT JOIN ratings AS rat ON b.book_id = rat.book_id
    GROUP BY b.book_id, b.title
    '''
run_query(query)

,book_id,title,qtd_avaliacoes,classificacao_media
0,1,'Salem's Lot,2,3.666667
1,2,1 000 Places to See Before You Die,1,2.500000
2,3,13 Little Blue Envelopes (Little Blue Envelope...,3,4.666667
3,4,1491: New Revelations of the Americas Before C...,2,4.500000
4,5,1776,4,4.000000
...,...,...,...,...
995,996,Wyrd Sisters (Discworld #6; Witches #2),3,3.666667
996,997,Xenocide (Ender's Saga #3),3,3.400000
997,998,Year of Wonders,4,3.200000
998,999,You Suck (A Love Story #2),2,4.500000


In [6]:
resultado = run_query(query)
print('NULL por coluna:')
print(resultado.isna().sum())
print('\nLivros com zero avaliações:', (resultado['qtd_avaliacoes'] == 0).sum())

NULL por coluna:
book_id                0
title                  0
qtd_avaliacoes         0
classificacao_media    0
dtype: int64

Livros com zero avaliações: 6


**Resultado e conclusão:** 1000 livros (catálogo completo). 6 sem nenhuma avaliação em texto — mas todos os 1000 têm nota (zero NULL em `classificacao_media`). Ou seja: classificar e escrever review são engajamentos independentes, e o texto é o canal mais escasso. Retê-los só foi possível pelo `LEFT JOIN`.

## Diário de Decisões Analíticas

Registro das escolhas metodológicas sob meu julgamento. Passos sem entrada não exigiram decisão — foram execução direta do enunciado.

**Passo 3 — [autoral]** Interpretei "depois de 1 de janeiro de 2000" como `>` (exclusivo). Livros de 2000-01-01 ficam fora. Resultado: 819.

**Passo 4 — [autoral]** Optei por `LEFT JOIN` em vez de `INNER`. A verificação confirmou 6 livros sem avaliação em texto — com `INNER JOIN` sobre reviews eles cairiam e o resultado seria 994, não 1000. O `LEFT` foi decisão necessária para preservar o catálogo completo.